# 02 Experiments (protocol v2): re-running the method described in the dissertation

Runs in Google Colab (GPU recommended). Implements dissertation Chapter 4 as the specification, with a clean protocol:

1. **Training pool** = DAIC-WOZ train + dev (140). **Held-out**: DAIC-WOZ official test (46) and E-DAIC (86 participants in no DAIC-WOZ split; its
   official test split of 56 is also scored as its own cohort), each scored once.
2. Architecture and graph hyperparameters fixed at the values the dissertation reports (Table 9). Optuna (TPE) tunes
   learning rate, weight decay, epochs and the class-weight multiplier m, by repeated stratified 5-fold CV inside the pool.
3. **Cross-fitted ensemble**: 5 folds x 4 seeds = 20 members. The threshold is calibrated on the out-of-fold probabilities
   of these same members (fewest false negatives with false-positive rate <= 6/21).

v1 (results in `results/experiments/`) tuned 9 hyperparameters on 107 participants and calibrated the threshold on
different models than the ones that scored the cohorts; its operating points were unstable. v2 results go to
`results/experiments/v2/`.

Deliberate changes from the legacy code: paralinguistic tags counted correctly, no 0.5 s turn filter, tags removed from the
lexical view, psycholinguistic view from `data/mapping/phq8_dsm5_empath.json`.
A controlled transcription analysis scores the DAIC-WOZ test participants on their E-DAIC ASR transcripts;
it is written to separate files (`controlled_*`) and is never part of the held-out evaluation.
Configuration frozen in `results/paper1/FROZEN.md` before scoring the E-DAIC cohorts.
Only aggregates are written to the repository. Per-participant outputs go to `OUTPUT_DIR`, outside it.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, sys
REPO = '/content/drive/MyDrive/05_Mestrado/Repositórios/depression-gnn-local/depression-gnn'
os.environ['DAIC_DATA_DIR'] = '/content/drive/MyDrive/Colab Notebooks/datasets'
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/outputs/paper1/v2'   # outside the repository
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, REPO)

In [3]:
!pip install -q -r requirements.txt
!python -m pytest -q tests

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
....................................                                     [100%]
=============================== warnings summary ===============================
tests/test_experiment.py::test_run_final[inductgcn-words-views0]
  /content/drive/MyDrive/05_Mestrado/Repositórios/depression-gnn-local/depression-gnn/src/paper1_graph_model/models.py:53: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
    a = torch.sparse_coo_tensor(edge_index, edge_weight, (x.shape[0], x.shape[0]))

tests/test_experiment.p

## Configuration

In [4]:
import torch
N_TRIALS = 30          # Optuna trials per model (restricted space: 4 hyperparameters)
N_FOLDS = 5
N_REPEATS = 2          # repeated CV inside the objective
SEEDS_PER_FOLD = 4     # ensemble = N_FOLDS * SEEDS_PER_FOLD = 20 members
SEED = 42
MAX_FPR = 6 / 21
SPACE = 'restricted'   # or 'full' for all of dissertation Table 9
STUDY_VERSION = 'v2.1'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
STORAGE = f'sqlite:///{OUTPUT_DIR}/optuna.db'   # resumable if the runtime disconnects
print(DEVICE, torch.__version__)

cuda 2.14.1+cu130


## Cohorts and features

In [5]:
from src.common import data
from src.common.features import EmpathScorer
from src.paper1_graph_model.corpus import build_cohorts
from src.paper1_graph_model.experiment import merge_cohorts

scorer = EmpathScorer()
cohorts = build_cohorts(data.data_root(), scorer, psycholinguistic='dsm5')
pool = merge_cohorts('daic_pool', cohorts['daic_train'], cohorts['daic_dev'])
held_out = [cohorts['daic_test'], cohorts['edaic'], cohorts['edaic_test']]
controlled = cohorts['asr_daic_test']   # controlled transcription analysis: never in held_out
for c in [pool] + held_out + [controlled]:
    print(f'{c.name:12s} n={len(c.y):4d} positives={int(c.y.sum()):3d}')

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


daic_pool    n= 140 positives= 42
daic_test    n=  46 positives= 14
edaic        n=  86 positives= 29
edaic_test   n=  56 positives= 21
asr_daic_test n=  46 positives= 14


## Models (rows of dissertation Table 11)

In [6]:
from src.paper1_graph_model.experiment import ModelSpec
ALL = ('temporal', 'paralinguistic', 'psycholinguistic')
CATS = scorer.categories
SPECS = [
    ModelSpec('InductGCN',                      'inductgcn', 'words',  ()),
    ModelSpec('GAT text-only',                  'gat',       'words',  ()),
    ModelSpec('GCN+GAT hybrid',                 'gcn_gat',   'words',  ALL),
    ModelSpec('EmpathGCN',                      'gcn',       'empath', ('psycholinguistic',), CATS),
    ModelSpec('GAT-GCN-FeExtr (Empath graph)',  'gat',       'empath', ALL, CATS),
    ModelSpec('GAT-GCN-FeExtr (word graph)',    'gat',       'words',  ALL),
]

## Hyperparameter search (training pool only)

In [7]:
import json, pathlib
from src.paper1_graph_model.experiment import tune, hp_from_params
RESULTS = pathlib.Path('results/experiments/v2'); RESULTS.mkdir(parents=True, exist_ok=True)

best = {}
for spec in SPECS:
    study = tune(spec, pool, N_TRIALS, N_FOLDS, N_REPEATS, SEED, DEVICE, storage=STORAGE,
                 space=SPACE, study_name=f'{spec.name} [{STUDY_VERSION}]')
    best[spec.name] = {'params': study.best_params, 'cv_pr_auc': study.best_value,
                       'n_trials': len(study.trials), 'space': SPACE}
    print(f'{spec.name:32s} CV PR-AUC={study.best_value:.3f}  {study.best_params}')
(RESULTS / 'best_params.json').write_text(json.dumps(best, indent=2))

[I 2026-11-25 04:17:53,803] Using an existing study with `study_name='InductGCN [v2.1]'` instead of creating a new one.
[I 2026-11-25 04:17:53,931] Using an existing study with `study_name='GAT text-only [v2.1]'` instead of creating a new one.
[I 2026-11-25 04:17:54,042] Using an existing study with `study_name='GCN+GAT hybrid [v2.1]'` instead of creating a new one.


InductGCN                        CV PR-AUC=0.519  {'lr': 0.0004059611610484307, 'weight_decay': 0.0005603803105930283, 'epochs': 450, 'm': 2.8736874205941256}
GAT text-only                    CV PR-AUC=0.474  {'lr': 0.0002051338263087451, 'weight_decay': 0.00010255552094216992, 'epochs': 300, 'm': 4.598528437324806}


[I 2026-11-25 04:17:54,160] Using an existing study with `study_name='EmpathGCN [v2.1]'` instead of creating a new one.
[I 2026-11-25 04:17:54,288] Using an existing study with `study_name='GAT-GCN-FeExtr (Empath graph) [v2.1]'` instead of creating a new one.


GCN+GAT hybrid                   CV PR-AUC=0.493  {'lr': 0.00020057247980860625, 'weight_decay': 0.0038151909356390163, 'epochs': 450, 'm': 4.842623593702834}
EmpathGCN                        CV PR-AUC=0.638  {'lr': 0.005397240148364171, 'weight_decay': 0.00016309127624404912, 'epochs': 400, 'm': 2.468648311640725}


[I 2026-11-25 04:17:54,405] Using an existing study with `study_name='GAT-GCN-FeExtr (word graph) [v2.1]'` instead of creating a new one.


GAT-GCN-FeExtr (Empath graph)    CV PR-AUC=0.582  {'lr': 0.002411713410724161, 'weight_decay': 0.00011709584632677918, 'epochs': 600, 'm': 3.288036619899568}
GAT-GCN-FeExtr (word graph)      CV PR-AUC=0.498  {'lr': 0.0001794662726839833, 'weight_decay': 0.000981709307142642, 'epochs': 350, 'm': 3.0187305646600144}


1602

## Cross-fitted ensembles, scored once on each held-out cohort

In [8]:
import numpy as np, pandas as pd
from src.paper1_graph_model.experiment import run_final

def row(model, cohort, m):
    lo, hi = m['f1_macro_ci']
    return {'model': model, 'cohort': cohort, 'n': m['n'], 'positives': m['positives'],
            'tau': round(m['threshold'], 2), 'F1 macro': round(m['f1_macro'], 4),
            'F1 95% CI': f'[{lo:.3f}, {hi:.3f}]', 'recall': round(m['recall'], 3),
            'precision': round(m['precision'], 3), 'specificity': round(m['specificity'], 3),
            'ROC-AUC': round(m.get('roc_auc', np.nan), 3), 'PR-AUC': round(m.get('pr_auc', np.nan), 3),
            'FN': m['fn'], 'FP': m['fp']}

def rows_of(name, res):
    out = [row(name, 'pool (out-of-fold)', res['oof_metrics'])]
    return out + [row(name, c, m) for c, m in res['metrics'].items()]

results, rows = {}, []
for spec in SPECS:
    hp = hp_from_params(spec, best[spec.name]['params'])
    res = run_final(spec, hp, pool, held_out, N_FOLDS, SEEDS_PER_FOLD, SEED, MAX_FPR, DEVICE)
    results[spec.name] = res
    rows += rows_of(spec.name, res)
    print(spec.name, 'tau =', res['tau'], res['tau_info'])
metrics = pd.DataFrame(rows)
metrics.to_csv(RESULTS / 'metrics.csv', index=False)
metrics

/content/drive/MyDrive/05_Mestrado/Repositórios/depression-gnn-local/depression-gnn/src/paper1_graph_model/models.py:53: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
  a = torch.sparse_coo_tensor(edge_index, edge_weight, (x.shape[0], x.shape[0]))


InductGCN tau = 0.6 {'fn': 18, 'fp': 30, 'feasible': False, 'max_fpr': 0.2857142857142857}
GAT text-only tau = 0.6 {'fn': 18, 'fp': 30, 'feasible': False, 'max_fpr': 0.2857142857142857}
GCN+GAT hybrid tau = 0.6 {'fn': 17, 'fp': 34, 'feasible': False, 'max_fpr': 0.2857142857142857}
EmpathGCN tau = 0.46 {'fn': 21, 'fp': 26, 'feasible': True, 'max_fpr': 0.2857142857142857}
GAT-GCN-FeExtr (Empath graph) tau = 0.5 {'fn': 20, 'fp': 25, 'feasible': True, 'max_fpr': 0.2857142857142857}
GAT-GCN-FeExtr (word graph) tau = 0.6 {'fn': 18, 'fp': 28, 'feasible': True, 'max_fpr': 0.2857142857142857}


,model,cohort,n,positives,tau,F1 macro,F1 95% CI,recall,precision,specificity,ROC-AUC,PR-AUC,FN,FP
0,InductGCN,pool (out-of-fold),140,42,0.60,0.6196,"[0.533, 0.698]",0.571,0.444,0.694,0.676,0.532,18,30
1,InductGCN,daic_test,46,14,0.60,0.6734,"[0.518, 0.802]",0.714,0.500,0.688,0.650,0.482,4,10
2,InductGCN,edaic,86,29,0.60,0.5425,"[0.434, 0.646]",0.483,0.389,0.614,0.545,0.406,15,22
3,InductGCN,edaic_test,56,21,0.60,0.5784,"[0.442, 0.711]",0.571,0.462,0.600,0.589,0.502,9,14
4,GAT text-only,pool (out-of-fold),140,42,0.60,0.6196,"[0.536, 0.695]",0.571,0.444,0.694,0.681,0.537,18,30
5,GAT text-only,daic_test,46,14,0.60,0.5984,"[0.450, 0.732]",0.571,0.421,0.656,0.636,0.460,6,11
6,GAT text-only,edaic,86,29,0.60,0.5137,"[0.403, 0.616]",0.379,0.355,0.649,0.520,0.366,18,20
7,GAT text-only,edaic_test,56,21,0.60,0.5417,"[0.406, 0.662]",0.524,0.423,0.571,0.577,0.456,10,15
8,GCN+GAT hybrid,pool (out-of-fold),140,42,0.60,0.6051,"[0.524, 0.681]",0.595,0.424,0.653,0.674,0.556,17,34
9,GCN+GAT hybrid,daic_test,46,14,0.60,0.6349,"[0.480, 0.760]",0.714,0.455,0.625,0.658,0.479,4,12


In [9]:
# Per-participant probabilities stay outside the repository.
for name, res in results.items():
    for c in held_out:
        out = pd.DataFrame({'Participant_ID': c.ids, 'label': c.y, 'prob': res['probs'][c.name]})
        out.to_csv(f"{OUTPUT_DIR}/probs_{name.replace(' ', '_')}_{c.name}.csv", index=False)

In [10]:
# Controlled transcription analysis, kept apart from the held-out evaluation (separate files and keys):
# the DAIC-WOZ test participants scored on their E-DAIC ASR transcripts by the same ensembles and thresholds.
from src.paper1_graph_model.experiment import score_cohort
ctrl_rows = []
for spec in SPECS:
    sc = score_cohort(results[spec.name], spec, controlled, DEVICE)
    ctrl_rows.append(row(spec.name, controlled.name, sc['metrics']))
    pd.DataFrame({'Participant_ID': controlled.ids, 'label': controlled.y, 'prob': sc['probs']}).to_csv(
        f"{OUTPUT_DIR}/controlled_probs_{spec.name.replace(' ', '_')}_{controlled.name}.csv", index=False)
controlled_metrics = pd.DataFrame(ctrl_rows)
controlled_metrics.to_csv(RESULTS / 'controlled_asr_daic_test.csv', index=False)
controlled_metrics

,model,cohort,n,positives,tau,F1 macro,F1 95% CI,recall,precision,specificity,ROC-AUC,PR-AUC,FN,FP
0,InductGCN,asr_daic_test,46,14,0.60,0.6734,"[0.521, 0.802]",0.714,0.500,0.688,0.650,0.481,4,10
1,GAT text-only,asr_daic_test,46,14,0.60,0.6349,"[0.480, 0.760]",0.714,0.455,0.625,0.643,0.475,4,12
2,GCN+GAT hybrid,asr_daic_test,46,14,0.60,0.6157,"[0.470, 0.751]",0.714,0.435,0.594,0.656,0.459,4,13
3,EmpathGCN,asr_daic_test,46,14,0.46,0.5893,"[0.435, 0.720]",0.643,0.409,0.594,0.703,0.475,5,13
4,GAT-GCN-FeExtr (Empath graph),asr_daic_test,46,14,0.50,0.6538,"[0.504, 0.792]",0.571,0.500,0.750,0.743,0.467,6,8
5,GAT-GCN-FeExtr (word graph),asr_daic_test,46,14,0.60,0.6541,"[0.502, 0.781]",0.714,0.476,0.656,0.683,0.477,4,11


## Ablation of the professional-feature views (main model, hyperparameters fixed)

In [11]:
from dataclasses import replace
MAIN = 'GAT-GCN-FeExtr (Empath graph)'   # change after looking at the table above if needed
main = next(s for s in SPECS if s.name == MAIN)
hp_main = hp_from_params(main, best[main.name]['params'])
ablation = []
for drop in ALL + (None,):
    views = tuple(v for v in ALL if v != drop)
    spec = replace(main, name=f'without {drop}' if drop else 'all views', views=views)
    ablation += rows_of(spec.name, run_final(spec, hp_main, pool, held_out, N_FOLDS, SEEDS_PER_FOLD, SEED, MAX_FPR, DEVICE))
ablation = pd.DataFrame(ablation)
ablation.to_csv(RESULTS / 'ablation_views.csv', index=False)
ablation

,model,cohort,n,positives,tau,F1 macro,F1 95% CI,recall,precision,specificity,ROC-AUC,PR-AUC,FN,FP
0,without temporal,pool (out-of-fold),140,42,0.53,0.6234,"[0.533, 0.702]",0.524,0.458,0.735,0.662,0.494,20,26
1,without temporal,daic_test,46,14,0.53,0.5820,"[0.430, 0.726]",0.786,0.407,0.500,0.672,0.393,3,16
2,without temporal,edaic,86,29,0.53,0.6109,"[0.501, 0.713]",0.621,0.462,0.632,0.680,0.560,11,21
3,without temporal,edaic_test,56,21,0.53,0.6680,"[0.533, 0.785]",0.667,0.560,0.686,0.710,0.642,7,11
4,without paralinguistic,pool (out-of-fold),140,42,0.52,0.6317,"[0.547, 0.715]",0.571,0.462,0.714,0.646,0.489,18,28
5,without paralinguistic,daic_test,46,14,0.52,0.7310,"[0.598, 0.847]",0.929,0.542,0.656,0.799,0.528,1,11
6,without paralinguistic,edaic,86,29,0.52,0.6558,"[0.556, 0.762]",0.655,0.514,0.684,0.717,0.608,10,18
7,without paralinguistic,edaic_test,56,21,0.52,0.7049,"[0.576, 0.819]",0.714,0.600,0.714,0.737,0.638,6,10
8,without psycholinguistic,pool (out-of-fold),140,42,0.60,0.6092,"[0.520, 0.690]",0.429,0.462,0.786,0.629,0.530,24,21
9,without psycholinguistic,daic_test,46,14,0.60,0.7433,"[0.586, 0.863]",0.643,0.643,0.844,0.839,0.609,5,5


## DSM-5 mapping vs. legacy Empath domain groups (EmpathGCN and main model)

Rebuilds the features with the legacy `EMPATH_DOMAIN_GROUPS` and repeats the final runs with the same hyperparameters.

In [12]:
cohorts_dom = build_cohorts(data.data_root(), scorer, psycholinguistic='domains')
pool_dom = merge_cohorts('daic_pool', cohorts_dom['daic_train'], cohorts_dom['daic_dev'])
held_dom = [cohorts_dom['daic_test'], cohorts_dom['edaic']]
cmp_rows = []
for name in ['EmpathGCN', MAIN]:
    spec = next(s for s in SPECS if s.name == name)
    hp = hp_from_params(spec, best[name]['params'])
    res_dom = run_final(spec, hp, pool_dom, held_dom, N_FOLDS, SEEDS_PER_FOLD, SEED, MAX_FPR, DEVICE)
    cmp_rows += rows_of(f'{name} / DSM-5 mapping', results[name]) + rows_of(f'{name} / legacy domains', res_dom)
mapping_vs_domains = pd.DataFrame(cmp_rows)
mapping_vs_domains.to_csv(RESULTS / 'mapping_vs_domains.csv', index=False)
mapping_vs_domains

,model,cohort,n,positives,tau,F1 macro,F1 95% CI,recall,precision,specificity,ROC-AUC,PR-AUC,FN,FP
0,EmpathGCN / DSM-5 mapping,pool (out-of-fold),140,42,0.46,0.6129,"[0.525, 0.692]",0.500,0.447,0.735,0.653,0.503,21,26
1,EmpathGCN / DSM-5 mapping,daic_test,46,14,0.46,0.5965,"[0.440, 0.732]",0.714,0.417,0.562,0.690,0.469,4,14
2,EmpathGCN / DSM-5 mapping,edaic,86,29,0.46,0.6210,"[0.514, 0.719]",0.621,0.474,0.649,0.620,0.461,11,20
3,EmpathGCN / DSM-5 mapping,edaic_test,56,21,0.46,0.6517,"[0.510, 0.779]",0.667,0.538,0.657,0.669,0.546,7,12
4,EmpathGCN / legacy domains,pool (out-of-fold),140,42,0.41,0.6477,"[0.560, 0.731]",0.595,0.481,0.724,0.663,0.514,17,27
5,EmpathGCN / legacy domains,daic_test,46,14,0.41,0.5965,"[0.440, 0.732]",0.714,0.417,0.562,0.701,0.466,4,14
6,EmpathGCN / legacy domains,edaic,86,29,0.41,0.5885,"[0.484, 0.684]",0.724,0.438,0.526,0.632,0.474,8,27
7,GAT-GCN-FeExtr (Empath graph) / DSM-5 mapping,pool (out-of-fold),140,42,0.50,0.6294,"[0.541, 0.715]",0.524,0.468,0.745,0.655,0.536,20,25
8,GAT-GCN-FeExtr (Empath graph) / DSM-5 mapping,daic_test,46,14,0.50,0.7262,"[0.589, 0.846]",0.857,0.545,0.688,0.815,0.564,2,10
9,GAT-GCN-FeExtr (Empath graph) / DSM-5 mapping,edaic,86,29,0.50,0.6392,"[0.521, 0.746]",0.483,0.538,0.789,0.720,0.606,15,12


## Attention over Empath categories (auditability)

Mean attention from each category node to each test participant, averaged over heads and the 20 members.
Per-participant matrices go to `OUTPUT_DIR`; only the class-level ranking is written to the repository.

In [13]:
from src.paper1_graph_model.experiment import attention_to_vocabulary
test_c = cohorts['daic_test']
att = attention_to_vocabulary(results[main.name], main, test_c, DEVICE)
att.to_csv(f"{OUTPUT_DIR}/attention_test_{main.name.replace(' ', '_')}.csv")
by_class = pd.DataFrame({'control': att[test_c.y == 0].mean(), 'depressed': att[test_c.y == 1].mean()})
by_class['delta'] = by_class['depressed'] - by_class['control']
by_class = by_class.sort_values('delta', ascending=False)
by_class.to_csv(RESULTS / 'attention_by_class_test.csv')
by_class.head(20)

,control,depressed,delta
neglect,0.004940,0.009527,0.004587
aggression,0.011273,0.015583,0.004311
furniture,0.002703,0.006540,0.003836
morning,0.007091,0.010707,0.003616
health,0.006982,0.010424,0.003441
gain,0.004417,0.007468,0.003051
masculine,0.006588,0.009422,0.002834
timidity,0.008706,0.011387,0.002681
royalty,0.003161,0.005552,0.002391
real_estate,0.006921,0.009309,0.002388
